In [10]:
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────
# 종합 실습용 데이터 — 새 스냅샷 (이 셀부터 단독 실행 가능)
# ─────────────────────────────────────────────
np.random.seed(7)

n_cust = 200
cust = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_cust + 1)],
    "region": np.random.choice(["서울", "경기", "부산", "인천"], n_cust),
    "membership": np.random.choice(["basic", "premium", "vip"], n_cust, p=[0.6, 0.3, 0.1]),
})

# ── customers 랜덤 칼럼 추가 ──
cust["age"] = np.random.randint(20, 65, n_cust)
cust["gender"] = np.random.choice(["M", "F"], n_cust)
cust["join_date"] = pd.to_datetime("2023-01-01") + pd.to_timedelta(np.random.randint(0, 900, n_cust), unit="D")
cust["is_active"] = np.random.choice([True, False], n_cust, p=[0.85, 0.15])

cats = ["패션", "뷰티", "식품", "가전"]
n_prod = 30
prod = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_prod + 1)],
    "category": np.random.choice(cats, n_prod),
    "price": np.random.choice([12000, 25000, 40000, 75000], n_prod),
})

# ── products 랜덤 칼럼 추가 ──
prod["brand"] = np.random.choice(["A브랜드", "B브랜드", "C브랜드", "D브랜드"], n_prod)
prod["stock_qty"] = np.random.randint(0, 500, n_prod)
prod["rating"] = np.round(np.random.uniform(2.5, 5.0, n_prod), 1)
prod["is_discontinued"] = np.random.choice([True, False], n_prod, p=[0.1, 0.9])

n_ord = 1500
oc = np.random.choice(cust["customer_id"], n_ord)
op = np.random.choice(prod["product_id"], n_ord)
qty = np.random.choice([1, 1, 2, 3], n_ord)
amt = prod.set_index("product_id")["price"].loc[op].values * qty
odate = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 150, n_ord), unit="D")
ordr = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_ord + 1)],
    "customer_id": oc, "product_id": op,
    "quantity": qty, "amount": amt.astype(float), "order_date": odate,
})

# ── orders 랜덤 칼럼 추가 ──
ordr["payment_method"] = np.random.choice(["card", "cash", "point", "bank_transfer"], n_ord, p=[0.6, 0.15, 0.15, 0.1])
ordr["channel"] = np.random.choice(["web", "app", "offline"], n_ord, p=[0.4, 0.5, 0.1])
ordr["discount_rate"] = np.random.choice([0, 0.05, 0.1, 0.2], n_ord, p=[0.5, 0.25, 0.15, 0.1])
ordr["is_cancelled"] = np.random.choice([True, False], n_ord, p=[0.05, 0.95])

print("스냅샷 준비 완료 — orders:", ordr.shape, "| customers:", cust.shape, "| products:", prod.shape)

스냅샷 준비 완료 — orders: (1500, 10) | customers: (200, 7) | products: (30, 7)


In [14]:
from IPython.display import display, Markdown

# ─────────────────────────────────────────────
# 시나리오 1 — 검증 → 병합
# ─────────────────────────────────────────────

# (1) 검증: 룩업 표의 키가 유일한가, 주문의 키가 모두 매칭되는가
print("[병합 전 검증]")
cust_dup = cust["customer_id"].duplicated().sum()
prod_dup = prod["product_id"].duplicated().sum()
cust_unmatched = (~ordr["customer_id"].isin(cust["customer_id"])).sum()
prod_unmatched = (~ordr["product_id"].isin(prod["product_id"])).sum()

print("  customers 키 중복:", cust_dup, "건")
print("  products  키 중복:", prod_dup, "건")
print("  매칭 안 되는 customer_id:", cust_unmatched, "건")
print("  매칭 안 되는 product_id :", prod_unmatched, "건")

# 세 조건(중복 없음 + 두 키 모두 완전 매칭)이 전부 참이어야 m:1 병합이 안전
is_safe = (cust_dup == 0) and (prod_dup == 0) and (cust_unmatched == 0) and (prod_unmatched == 0)

md = f"""
### 키 검증 결과

| 검증 항목 | 결과 |
|---|---|
| customers 키 중복 | {cust_dup}건 |
| products 키 중복 | {prod_dup}건 |
| 매칭 안 되는 customer_id | {cust_unmatched}건 |
| 매칭 안 되는 product_id | {prod_unmatched}건 |

**검증 결과 {"두 룩업 테이블의 키가 모두 유일하고, 주문 테이블의 외래키가 100% 매칭되므로" if is_safe else "일부 키 중복 또는 미매칭이 존재하므로"} m:1 병합이 {"안전하다" if is_safe else "위험할 수 있다"}.**
"""
display(Markdown(md))

# (2) 병합: validate로 관계 가정(m:1)을 강제. indicator로 매칭 확인.
df = (
    ordr
    .merge(cust, on="customer_id", how="left", validate="m:1", indicator="_merge_cust")
    .merge(prod, on="product_id", how="left", validate="m:1", indicator="_merge_prod")
)

print("\n[병합 결과] 행 수:", len(df), "(원본 주문 수와 같으면 폭증 없음)")
print("  _merge_cust 매칭 분포:\n", df["_merge_cust"].value_counts())
print("  _merge_prod 매칭 분포:\n", df["_merge_prod"].value_counts())

# 확인 끝났으면 indicator 칼럼은 제거
df = df.drop(columns=["_merge_cust", "_merge_prod"])
display(df.head(3))

[병합 전 검증]
  customers 키 중복: 0 건
  products  키 중복: 0 건
  매칭 안 되는 customer_id: 0 건
  매칭 안 되는 product_id : 0 건



### 키 검증 결과

| 검증 항목 | 결과 |
|---|---|
| customers 키 중복 | 0건 |
| products 키 중복 | 0건 |
| 매칭 안 되는 customer_id | 0건 |
| 매칭 안 되는 product_id | 0건 |

**검증 결과 두 룩업 테이블의 키가 모두 유일하고, 주문 테이블의 외래키가 100% 매칭되므로 m:1 병합이 안전하다.**



[병합 결과] 행 수: 1500 (원본 주문 수와 같으면 폭증 없음)
  _merge_cust 매칭 분포:
 _merge_cust
both          1500
left_only        0
right_only       0
Name: count, dtype: int64
  _merge_prod 매칭 분포:
 _merge_prod
both          1500
left_only        0
right_only       0
Name: count, dtype: int64


,order_id,customer_id,product_id,quantity,amount,order_date,payment_method,channel,discount_rate,is_cancelled,...,age,gender,join_date,is_active,category,price,brand,stock_qty,rating,is_discontinued
0,O00001,C0137,P013,1,40000.0,2025-01-16,point,web,0.2,False,...,51,M,2025-01-14,True,뷰티,40000,C브랜드,171,4.9,False
1,O00002,C0194,P002,3,36000.0,2025-04-08,cash,app,0.0,False,...,56,M,2024-08-22,True,패션,12000,A브랜드,297,2.7,False
2,O00003,C0160,P021,3,120000.0,2025-05-27,card,app,0.2,False,...,50,M,2025-05-28,False,가전,40000,A브랜드,130,2.5,False


In [15]:
# ─────────────────────────────────────────────
# 시나리오 2 — 월별 KPI (기존 코드)
# ─────────────────────────────────────────────
df["month"] = df["order_date"].dt.to_period("M").astype(str)

monthly_kpi = (
    df.groupby("month")
    .agg(
        총매출=("amount", "sum"),
        주문건수=("order_id", "count"),
        객단가=("amount", "mean"),
        구매고객수=("customer_id", "nunique"),
    )
    .round(0)
)
monthly_kpi["매출증감률(%)"] = (monthly_kpi["총매출"].pct_change() * 100).round(1)

print("[월별 KPI 요약표]")
display(monthly_kpi)

# ─────────────────────────────────────────────
# (1) 지역 × 회원등급 매출 교차표 — pivot_table
# ─────────────────────────────────────────────
region_membership_pivot = pd.pivot_table(
    df,
    index="region",
    columns="membership",
    values="amount",
    aggfunc="sum",
    fill_value=0,        # 해당 조합 데이터가 없으면 0으로
    margins=True,         # 행/열 합계 추가
    margins_name="합계",
).round(0)

print("\n[지역 x 회원등급 매출 교차표]")
display(region_membership_pivot)

# ─────────────────────────────────────────────
# (2) 카테고리별 KPI 요약표 — groupby + agg
# ─────────────────────────────────────────────
category_kpi = (
    df.groupby("category")
    .agg(
        총매출=("amount", "sum"),
        주문건수=("order_id", "count"),
        객단가=("amount", "mean"),
    )
    .round(0)
    .sort_values("총매출", ascending=False)   # 매출 큰 순으로 정렬
)

print("\n[카테고리별 KPI 요약표]")
display(category_kpi)



[월별 KPI 요약표]


,총매출,주문건수,객단가,구매고객수,매출증감률(%)
month,,,,,
2025-01,17598000.0,327,53817.0,158,NaN
2025-02,16019000.0,291,55048.0,156,-9.0
2025-03,18504000.0,317,58372.0,168,15.5
2025-04,14110000.0,276,51123.0,150,-23.7
2025-05,15058000.0,289,52104.0,146,6.7



[지역 x 회원등급 매출 교차표]


membership,basic,premium,vip,합계
region,,,,
경기,12362000.0,2926000.0,1911000.0,17199000.0
부산,11502000.0,5822000.0,1609000.0,18933000.0
서울,11519000.0,8203000.0,2285000.0,22007000.0
인천,14072000.0,8640000.0,438000.0,23150000.0
합계,49455000.0,25591000.0,6243000.0,81289000.0



[카테고리별 KPI 요약표]


,총매출,주문건수,객단가
category,,,
가전,28187000.0,545,51719.0
뷰티,21200000.0,341,62170.0
식품,17569000.0,362,48533.0
패션,14333000.0,252,56877.0


In [ ]:
# ─────────────────────────────────────────────
# 특성 추가 — transform: 각 주문이 '자신이 속한 지역'의
#                       총매출에서 차지하는 비중(%)
# ─────────────────────────────────────────────

df["지역총매출"] = df.groupby("region")["amount"].transform("sum")
df["지역내_매출비중(%)"] = (df["amount"] / df["지역총매출"] * 100).round(2)

# 지역별로 정렬해서 확인
result = df[["order_id", "region", "amount", "지역총매출", "지역내_매출비중(%)"]].sort_values("region")
display(result.head(10))

print("\n지역별 비중 합계 검산 (반올림 오차로 100에서 소폭 벗어날 수 있음):")
print(df.groupby("region")["지역내_매출비중(%)"].sum())

,order_id,region,amount,지역총매출,지역내_매출비중(%)
1162,O01163,경기,40000.0,17199000.0,0.23
310,O00311,경기,36000.0,17199000.0,0.21
1409,O01410,경기,40000.0,17199000.0,0.23
308,O00309,경기,24000.0,17199000.0,0.14
758,O00759,경기,24000.0,17199000.0,0.14
1412,O01413,경기,36000.0,17199000.0,0.21
759,O00760,경기,25000.0,17199000.0,0.15
761,O00762,경기,75000.0,17199000.0,0.44
300,O00301,경기,40000.0,17199000.0,0.23
766,O00767,경기,12000.0,17199000.0,0.07



지역별 비중 합계 검산 (반올림 오차로 100에서 소폭 벗어날 수 있음):
region
경기    100.43
부산     99.68
서울     99.42
인천     99.95
Name: 지역내_매출비중(%), dtype: float64


In [18]:
# ─────────────────────────────────────────────
# 핵심 숫자 자동 추출 (보고서 문장에 채워 넣기 위함)
# ─────────────────────────────────────────────

# 월별
best_month = monthly_kpi["총매출"].idxmax()
best_month_sales = monthly_kpi["총매출"].max()
worst_month = monthly_kpi["총매출"].idxmin()
worst_month_sales = monthly_kpi["총매출"].min()

# 카테고리 / 지역 / 회원등급 / 채널 / 결제수단 / 브랜드
top_category = df.groupby("category")["amount"].sum().idxmax()
top_category_sales = df.groupby("category")["amount"].sum().max()

top_region = df.groupby("region")["amount"].sum().idxmax()
top_region_sales = df.groupby("region")["amount"].sum().max()

top_membership = df.groupby("membership")["amount"].sum().idxmax()
top_membership_sales = df.groupby("membership")["amount"].sum().max()

top_channel = df.groupby("channel")["amount"].sum().idxmax()
top_payment = df.groupby("payment_method")["amount"].sum().idxmax()
top_brand = df.groupby("brand")["amount"].sum().idxmax()

# 전체 요약 지표
overall_sales = df["amount"].sum()
overall_aov = df["amount"].mean()
total_orders = len(df)
total_customers = df["customer_id"].nunique()

# 운영 지표 (취소율 / 할인율 / VIP 비중)
cancel_rate = df["is_cancelled"].mean() * 100
avg_discount = df["discount_rate"].mean() * 100
vip_ratio = (cust["membership"] == "vip").mean() * 100

print("자동 추출된 핵심 숫자")
print(f"  최대 매출 월     : {best_month} ({best_month_sales:,.0f}원)")
print(f"  최소 매출 월     : {worst_month} ({worst_month_sales:,.0f}원)")
print(f"  최대 매출 카테고리 : {top_category} ({top_category_sales:,.0f}원)")
print(f"  최대 매출 지역   : {top_region} ({top_region_sales:,.0f}원)")
print(f"  최대 매출 회원등급 : {top_membership} ({top_membership_sales:,.0f}원)")
print(f"  최대 매출 채널   : {top_channel}")
print(f"  최대 매출 결제수단 : {top_payment}")
print(f"  최대 매출 브랜드  : {top_brand}")
print(f"  전체 매출      : {overall_sales:,.0f}원")
print(f"  전체 객단가     : {overall_aov:,.0f}원")
print(f"  전체 주문 건수   : {total_orders:,}건")
print(f"  구매 고객 수    : {total_customers:,}명")
print(f"  평균 할인율     : {avg_discount:.1f}%")
print(f"  주문 취소율     : {cancel_rate:.1f}%")
print(f"  VIP 고객 비중   : {vip_ratio:.1f}%")

자동 추출된 핵심 숫자
  최대 매출 월     : 2025-03 (18,504,000원)
  최소 매출 월     : 2025-04 (14,110,000원)
  최대 매출 카테고리 : 가전 (28,187,000원)
  최대 매출 지역   : 인천 (23,150,000원)
  최대 매출 회원등급 : basic (49,455,000원)
  최대 매출 채널   : app
  최대 매출 결제수단 : card
  최대 매출 브랜드  : C브랜드
  전체 매출      : 81,289,000원
  전체 객단가     : 54,193원
  전체 주문 건수   : 1,500건
  구매 고객 수    : 200명
  평균 할인율     : 4.9%
  주문 취소율     : 5.1%
  VIP 고객 비중   : 9.0%


In [20]:
# ─────────────────────────────────────────────
# 회원등급별 1인당 구매력 비교 (총매출의 왜곡 보정)
# ─────────────────────────────────────────────
membership_summary = (
    df.groupby("membership")
    .agg(총매출=("amount", "sum"), 주문건수=("order_id", "count"))
)

# 핵심: 고객 수는 '주문에 등장한 횟수'가 아니라
#       고객 테이블(cust) 기준으로 세야 정확함 (주문 안 한 고객도 분모에 포함되어야 하므로)
membership_summary["고객수"] = cust.groupby("membership").size()
membership_summary["1인당매출"] = (membership_summary["총매출"] / membership_summary["고객수"]).round(0)
membership_summary["1인당주문건수"] = (membership_summary["주문건수"] / membership_summary["고객수"]).round(2)

membership_summary = membership_summary.round(0).sort_values("1인당매출", ascending=False)

print("[회원등급별 구매력 비교: 총매출 vs 1인당매출]")
display(membership_summary)

# 두 기준(총매출 vs 1인당매출)의 1위가 다른지 자동 체크
top_membership_by_total = membership_summary["총매출"].idxmax()
top_membership_by_per_head = membership_summary["1인당매출"].idxmax()

print(f"\n  총매출 기준 1위 등급   : {top_membership_by_total}")
print(f"  1인당매출 기준 1위 등급 : {top_membership_by_per_head}")

[회원등급별 구매력 비교: 총매출 vs 1인당매출]


,총매출,주문건수,고객수,1인당매출,1인당주문건수
membership,,,,,
premium,25591000.0,463,58,441224.0,8.0
basic,49455000.0,912,124,398831.0,7.0
vip,6243000.0,125,18,346833.0,7.0



  총매출 기준 1위 등급   : basic
  1인당매출 기준 1위 등급 : premium
